# 第 8 章实验：XOR、非线性与反向传播
本实验只使用内存合成点。修改下一格参数并从头运行，观察隐藏层、学习率和线性模型的限制。

In [ ]:
import numpy as np
import torch
from torch import nn
import matplotlib.pyplot as plt

seed = 8
hidden_dim = 8
learning_rate = 0.05
steps = 250
torch.manual_seed(seed)
np.random.seed(seed)

In [ ]:
base_x = torch.tensor([[0., 0.], [0., 1.], [1., 0.], [1., 1.]])
base_y = torch.tensor([0, 1, 1, 0])
x = base_x.repeat(32, 1) + 0.08 * torch.randn(128, 2)
y = base_y.repeat(32)
print('x:', tuple(x.shape), x.dtype, 'y:', tuple(y.shape), y.dtype)

In [ ]:
def train_model(model, n_steps=steps):
    optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)
    loss_fn = nn.CrossEntropyLoss()
    history = []
    for _ in range(n_steps):
        optimizer.zero_grad()
        logits = model(x)
        loss = loss_fn(logits, y)
        loss.backward()
        optimizer.step()
        history.append(float(loss.detach()))
    return history

mlp = nn.Sequential(nn.Linear(2, hidden_dim), nn.ReLU(), nn.Linear(hidden_dim, 2))
history = train_model(mlp)
with torch.no_grad():
    accuracy = (mlp(x).argmax(1) == y).float().mean().item()
print(f'MLP final loss={history[-1]:.4f}, accuracy={accuracy:.3f}')

In [ ]:
grid_x, grid_y = np.meshgrid(np.linspace(-0.3, 1.3, 120), np.linspace(-0.3, 1.3, 120))
grid = torch.tensor(np.c_[grid_x.ravel(), grid_y.ravel()], dtype=torch.float32)
with torch.no_grad():
    region = mlp(grid).argmax(1).reshape(grid_x.shape).numpy()
fig, axes = plt.subplots(1, 2, figsize=(9, 3.5))
axes[0].plot(history); axes[0].set(title='training loss', xlabel='step', ylabel='cross entropy')
axes[1].contourf(grid_x, grid_y, region, alpha=0.3, levels=[-0.5, 0.5, 1.5])
axes[1].scatter(x[:, 0], x[:, 1], c=y, s=12, cmap='coolwarm')
axes[1].set(title='MLP decision regions', xlim=(-0.3, 1.3), ylim=(-0.3, 1.3))
plt.tight_layout(); plt.show()

## 破坏实验：去掉隐藏层
线性分类器无法用一条直线分开 XOR。这里使用同一数据与训练步数作为对照。

In [ ]:
linear = nn.Linear(2, 2)
linear_history = train_model(linear)
with torch.no_grad():
    linear_accuracy = (linear(x).argmax(1) == y).float().mean().item()
print(f'linear final loss={linear_history[-1]:.4f}, accuracy={linear_accuracy:.3f}')
assert accuracy > linear_accuracy, '本次随机种子下 MLP 应明显优于线性模型'
print('观察结论：非线性隐藏层改变了可表达的决策边界。')